In [ ]:
import os
from pathlib import Path
import tempfile
import shutil

import teehr
import hvplot.pandas
# Tell Bokeh to output plots in the notebook
from bokeh.io import output_notebook
output_notebook()

teehr.__version__

In [ ]:
from teehr.evaluation.spark_session_utils import create_spark_session
spark = create_spark_session()

In [ ]:
ev = teehr.RemoteReadOnlyEvaluation(spark=spark)

In [ ]:
ev.configurations.to_pandas()

In [ ]:
USGS_GAGE_ID = "usgs-01010070"
PRIMARY_CONFIGURATION_NAME = "usgs_observations"
SECONDARY_CONFIGURATION_NAME = "nwm30_retrospective"

In [ ]:
primary_df = ev.primary_timeseries.filter([
    {
        "column": "configuration_name",
        "operator": "=",
        "value": PRIMARY_CONFIGURATION_NAME
    },
    {
        "column": "location_id",
        "operator": "=",
        "value": USGS_GAGE_ID
    }
]).to_pandas()
primary_df

In [ ]:
lxw = ev.location_crosswalks.filter([
    {
        "column": "primary_location_id",
        "operator": "=",
        "value": USGS_GAGE_ID
    }
]).to_pandas()
lxw

In [ ]:
lxw["secondary_location_id"].to_list()

In [ ]:
secondary_df = ev.secondary_timeseries.filter([
    {
        "column": "configuration_name",
        "operator": "=",
        "value": SECONDARY_CONFIGURATION_NAME
    },
    {
        "column": "location_id",
        "operator": "in",
        "value": lxw["secondary_location_id"].to_list()
    }
]).to_pandas()
secondary_df

In [ ]:
pt = primary_df.hvplot(
    x="value_time",
    y="value",
    label="USGS Observations",
)
st = secondary_df.hvplot(
    x="value_time",
    y="value",
    label="NWM3.0 Restrospective",
)
(st * pt).opts(width=700, height=400)

In [ ]:
joined_df = ev.table("sim_joined_timeseries").filter([
    {
        "column": "configuration_name",
        "operator": "=",
        "value": SECONDARY_CONFIGURATION_NAME
    },
    {
        "column": "secondary_location_id",
        "operator": "in",
        "value": lxw["secondary_location_id"].to_list()
    }
]).to_pandas()
joined_df

In [ ]:
pt = joined_df.hvplot(
    x="value_time",
    y="primary_value",
    label="USGS Observations",
)
st = joined_df.hvplot(
    x="value_time",
    y="secondary_value",
    label="NWM3.0 Restrospective",
)
(st * pt).opts(width=700, height=400)

In [ ]:
spark.stop()